<a href="https://colab.research.google.com/github/aldo02032004/social-media-kol-analysis/blob/main/kol_network_stance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install

In [4]:
import importlib.util
import torch

!pip uninstall -q -y great
!pip install -q "great @ git+https://github.com/azmkto/GREAT-Tools.git@6e126253da57688d951d1bb1d55441993c8842f3" emoji ftfy

CU_TAG = "cu124" if torch.cuda.is_available() else "cpu"
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{CU_TAG}
if CU_TAG == "cpu":
    print("Minimal runtime type adalah T4 GPU.")

missing = [m for m in ["great", "emoji", "ftfy", "openpyxl", "networkx", "scipy", "sklearn", "statsmodels", "huggingface_hub", "llama_cpp"]
           if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f"Gagal terpasang: {missing}")
print("Instalasi selesai.")

Instalasi selesai.


# 2. Import Library

In [5]:
import json, os, re, sys, unicodedata
from datetime import date            # tanggal buat prompt config
from functools import lru_cache

import emoji                         # emoji jadi teks
import networkx as nx                # jaringan, pagerank, louvain
import numpy as np
import pandas as pd
import torch
from google.colab import drive       # config, cache, label
from IPython.display import display
from scipy.sparse.linalg import eigsh   # eigenvector sparse
from sklearn.metrics import classification_report, cohen_kappa_score, f1_score, precision_score, recall_score
from statsmodels.stats.proportion import proportion_confint   # CI 95% proporsi
from tqdm.auto import tqdm

from great import EXPORT_COLUMNS, clean_for_bert, is_media_account
from great.labels import is_media_name

tqdm.pandas()

# 4. Form

In [6]:
PROJECT = "prabowo_vladivostok"   # nama file di Drive
TOPIC = "Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin"
STANCE_TARGET = "Presiden Prabowo dan pemerintah"
TARGET_ACCOUNTS = "prabowo, prabowosubianto, gerindra"   # akun target: tetep node, nggak ikut ranking
SOURCES_TEXT = "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"
REGENERATE_TOPIC_CONFIG = False

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
TOPIC, STANCE_TARGET = TOPIC.strip(), STANCE_TARGET.strip()
SOURCES = [s for s in re.split(r"[,\s]+", SOURCES_TEXT) if s]
assert PROJECT and TOPIC and STANCE_TARGET and SOURCES, "Form belum lengkap"

drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive"
EXPECTED_COLS = [c for c in EXPORT_COLUMNS if c != "Location"]

LLM_MODEL_REPO = "faizonly5953/gemma2-9b-cpt-sahabatai-v1-instruct-GGUF"
LLM_MODEL_FILE = "gemma2-9b-cpt-sahabatai-v1-instruct.q4_k_m.gguf"
LLM_N_CTX = 8192

INTERACTION_WEIGHT = {"retweet": 1.0, "reply": 0.5, "mention": 0.25}   # bisa dituning
SCORE_COLS = ["pagerank", "eigenvector", "total_engagement", "followers"]   # skor = rata-rata percentile
TOP_N_AUTHORS_PER_THEME = 10
MAX_POSTS_PER_AUTHOR_SUMMARY = 15
RANKING_MIN_CONFIDENCE = 0.5   # ambil dari hasil Evaluasi
COPYPASTA_MIN_ACCOUNTS = 3     # teks sama dari >= n akun = indikasi koordinasi

TOPIC_CONFIG_PATH = f"{DRIVE_DIR}/topic_config_{PROJECT}.json"
for opt in ("display.max_rows", "display.max_columns", "display.max_colwidth"):
    pd.set_option(opt, None)

Mounted at /content/drive


# 9. Load Gemma



In [7]:
import ctypes, glob, site

# wheel llama-cpp butuh CUDA 12, Colab udah CUDA 13 -> load manual
CUDA12_LIBS = ["libcudart.so.12", "libcublasLt.so.12", "libcublas.so.12"]


def find_lib(name):
    roots = {*site.getsitepackages(), *[p for p in sys.path if os.path.isdir(p)]}
    hits = [p for r in roots for p in glob.glob(f"{r}/nvidia/**/{name}", recursive=True)]
    return hits[0] if hits else None


if torch.cuda.is_available():
    if not all(map(find_lib, CUDA12_LIBS)):
        !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    missing_libs = [n for n in CUDA12_LIBS if not find_lib(n)]
    if missing_libs:
        raise RuntimeError(f"CUDA 12 nggak ketemu: {missing_libs}")
    for name in CUDA12_LIBS:
        ctypes.CDLL(find_lib(name), mode=ctypes.RTLD_GLOBAL)

for m in [m for m in sys.modules if m.startswith("llama_cpp")]:   # buang import gagal sebelumnya
    del sys.modules[m]

import llama_cpp
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    print("llama-cpp tanpa GPU, bakal lambat")
if "llm" not in globals():
    llm = Llama(model_path=hf_hub_download(repo_id=LLM_MODEL_REPO, filename=LLM_MODEL_FILE),   # ~5,8 GB
                n_gpu_layers=-1, n_ctx=LLM_N_CTX, n_batch=512, verbose=False)
print("Gemma siap")

gemma2-9b-cpt-sahabatai-v1-instruct.q4_k(…): reconstructing file:   0%|          |  0.00B / 5.76GB            

gemma2-9b-cpt-sahabatai-v1-instruct.q4_k(…): downloading bytes:           |  0.00B            

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Gemma siap


# Fungsi Gemma

In [8]:
def ask_llm(content, schema, max_tokens=700):
    r = llm.create_chat_completion(messages=[{"role": "user", "content": content}],
                                   response_format={"type": "json_object", "schema": schema},
                                   temperature=0, max_tokens=max_tokens)
    return json.loads(r["choices"][0]["message"]["content"])


def fmt_posts(posts, max_words=120):
    return "\n".join(f"{i}. " + " ".join(str(p).split()[:max_words]) for i, p in enumerate(posts, 1))

# Konfigurasi Gemma

In [9]:
CONFIG_SCHEMA = {
    "type": "object",
    "properties": {
        "themes": {"type": "array", "minItems": 2, "maxItems": 4,
                   "items": {"type": "object", "properties": {"nama": {"type": "string"}, "deskripsi": {"type": "string"}},
                             "required": ["nama", "deskripsi"]}},
        "keywords": {"type": "array", "minItems": 20, "maxItems": 60, "items": {"type": "string"}},
        "media_accounts": {"type": "array", "maxItems": 30, "items": {"type": "string"}},
    },
    "required": ["themes", "keywords", "media_accounts"],
}
TOPIC_CONFIG_PROMPT = f"""Kamu analis media sosial Indonesia yang menyiapkan konfigurasi klasifikasi otomatis.
Topik: "{TOPIC}". Tanggal hari ini: {date.today()}. Jangan menulis tahun di keyword kecuali yakin.

Cara pakai: cuitan tanpa satu pun keyword dibuang tanpa dibaca AI (keyword terlewat = cuitan relevan hilang,
terlalu umum = ribuan cuitan tidak relevan ikut dibaca). Cuitan yang lolos diberi satu tema. Akun di media_accounts dibuang.
Data mentah mencakup SEMUA cuitan yang menyebut tokoh utama, termasuk isu lain, jadi nama tokoh utama BUKAN penanda topik.

themes: 2-4 tema per sudut pandang percakapan (agenda/acara, kebijakan/kerja sama, reaksi politik), bukan per tokoh.
- nama: snake_case bahasa Indonesia, bukan "lainnya"/"umum"/"campuran".
- deskripsi: "Masuk: <1 kalimat>. Contoh frasa: <8-15 frasa khas, termasuk gaul/singkatan/julukan>. Bukan: <yang sering tertukar>."

keywords: 20-60 kata/frasa pendek. Kata >=5 huruf juga cocok sebagai bagian kata lain (hashtag gabungan).
- Isi: penanda khas topik ini (pihak lain, tempat, acara, instansi; ejaan Indonesia & Inggris, singkatan, typo umum).
- DILARANG: nama/julukan tokoh utama sendirian, dan kata umum lintas isu ("investasi", "pertahanan", "kesepakatan",
  "geopolitik", "infrastruktur", "energi", "komoditas", "digitalisasi", "kunjungan kerja", "kebijakan luar negeri").

media_accounts: username media/kanal resmi institusi terkait, tanpa @. Bukan akun pribadi, buzzer, influencer. Ragu = jangan.

Output HANYA JSON."""

topic_config = None
if os.path.exists(TOPIC_CONFIG_PATH) and not REGENERATE_TOPIC_CONFIG:
    with open(TOPIC_CONFIG_PATH, encoding="utf-8") as f:
        cached = json.load(f)
    if cached.get("_topic") == TOPIC:
        topic_config = cached

if topic_config is None:
    out = ask_llm(TOPIC_CONFIG_PROMPT, CONFIG_SCHEMA, max_tokens=2048)
    themes = {re.sub(r"\W+", "_", t["nama"].strip().lower()).strip("_"): t["deskripsi"].strip() for t in out["themes"]}
    themes.pop("lainnya", None)
    if not themes:
        raise RuntimeError("Gemma gagal bikin tema, jalanin ulang")
    topic_config = {"_topic": TOPIC, "themes": themes, "keywords": out["keywords"], "media_accounts": out["media_accounts"]}
    with open(TOPIC_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(topic_config, f, ensure_ascii=False, indent=2)

THEME_DESCRIPTIONS = {**topic_config["themes"], "lainnya": "topik di luar kategori di atas"}
THEMES = list(THEME_DESCRIPTIONS)
RELEVANT_THEMES = THEMES[:-1]
KW_RELEVAN = sorted({k.strip().lower() for k in topic_config["keywords"] if k.strip()}, key=len, reverse=True)
EXTRA_MEDIA_ACCOUNTS = {a.lower().lstrip("@") for a in topic_config["media_accounts"]}

RE_RELEVAN = re.compile("|".join(re.escape(k) if len(k) >= 5 else rf"(?<!\w){re.escape(k)}(?!\w)"
                                 for k in KW_RELEVAN)) if KW_RELEVAN else None
print(json.dumps(topic_config, ensure_ascii=False, indent=2))

{
  "themes": {
    "diplomasi_dan_geopolitik": "Masuk: Isu hubungan bilateral, politik luar negeri bebas aktif, dinamika geopolitik global, non-blok, hegemoni barat vs timur, serta sikap netralitas Indonesia. Contoh frasa: bebas aktif, poros rusia, geopolitik global, aliansi timur, hubungan diplomatik, non blok, penyeimbang kekuatan, blok barat, mitra strategis rusia, pertemuan bilateral, kedaulatan luar negeri, netralitas ri, multipolar. Bukan: Negosiasi spesifik pembelian alutsista atau perjanjian transfer energi nuklir.",
    "kerja_sama_ekonomi_energi_pertahanan": "Masuk: Kesepakatan konkret perdagangan, investasi industri, ketahanan pangan, pasokan pupuk, kerja sama energi nuklir, minyak bumi, alutsista, dan latihan militer bersama. Contoh frasa: pasokan gandum, impor pupuk, reaktor nuklir, rosatom, kerja sama alutsista, jet tempur su-35, transfer teknologi, investasi energi, impor minyak mentah, kerja sama pertahanan, latsitarda maritim, dedolarisasi perdagangan, brics payment. 

# 6. Load data dari Drive


In [10]:
def xlsx_url(link):
    match = re.search(r"/d/([\w-]+)", link)
    if not match:
        return link
    if "drive.google.com" in link:
        return f"https://drive.google.com/uc?export=download&id={match.group(1)}"
    return f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"


def load_raw_data(link):
    try:
        df = pd.read_excel(xlsx_url(link), header=1)
    except ValueError as e:
        raise ValueError(f"Gagal baca {link}, cek akses 'Anyone with the link'") from e
    df.columns = df.columns.str.strip()
    return df[EXPECTED_COLS]   # KeyError = kolom kurang


df_raw = pd.concat([load_raw_data(s).assign(source_file=f"file_{i}") for i, s in enumerate(SOURCES, 1)], ignore_index=True)
print(f"Data: {len(df_raw)} baris, {len(SOURCES)} file")

Data: 66891 baris, 1 file


# 7. Merge akun & filter media


In [24]:
ALIASES = {"kompastvcom": "kompascom", "hippoacademyid": "hippoacademy"}   # gabung manual
TWO_PART_SLD = {"co", "or", "ac", "go", "web", "my", "sch", "net", "com"}


@lru_cache(maxsize=None)   # abis edit ALIASES, jalanin ulang
def canonical(name):
    # beda ejaan akun (@kompascom, kompas.com) -> satu key
    if pd.isna(name):
        return None
    s = re.sub(r"^(https?://)?(www\.)?@?", "", unicodedata.normalize("NFKC", str(name)).strip().lower()).split("/")[0]
    parts = s.split(".")
    if len(parts) > 2 and re.fullmatch(r"[a-z]{2,3}", parts[-1]):   # domain, bukan handle IG bertitik
        parts = parts[-3:] if parts[-2] in TWO_PART_SLD and len(parts[-1]) == 2 else parts[-2:]
    key = re.sub(r"[^a-z0-9]", "", "".join(parts)) or re.sub(r"\W", "", s)   # non-latin tetep hurufnya
    return ALIASES.get(key, key) if key else None


assert canonical("@KOMPASTVCOM") == canonical("www.kompastv.com")
assert canonical("nasional.kompas.com") == canonical("@kompascom") == "kompascom"
assert canonical("https://www.republika.co.id/berita/x") == "republikacoid"
assert canonical("prabowo.subianto") == canonical("@Prabowo_Subianto") == "prabowosubianto"
assert canonical("𝐁𝐞𝐫𝐢𝐭𝐚 𝐈𝐧𝐝𝐨") == "beritaindo" and canonical("بيت") == "بيت" and canonical("🔥🔥") is None
assert canonical(float("nan")) is None and canonical("  ") is None

TARGET_IDS = {canonical(a) for a in TARGET_ACCOUNTS.split(",")} - {None}

In [12]:
NUM_COLS = ["Followers", "Retweeted", "Favourited"]
df_raw[NUM_COLS] = df_raw[NUM_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)
df_raw["author_key"] = df_raw["Author"].map(canonical)

is_media = pd.Series([is_media_account(a, p, extra_accounts=EXTRA_MEDIA_ACCOUNTS)
                      for a, p in zip(df_raw["Author"], df_raw["Media"])], index=df_raw.index)
is_media |= df_raw["Author"].map(is_media_name)
MEDIA_IDS = set(df_raw.loc[is_media, "author_key"].dropna()) | {canonical(a) for a in EXTRA_MEDIA_ACCOUNTS}
is_media |= df_raw["author_key"].isin(MEDIA_IDS)   # ejaan lain akun media

df_raw = df_raw[~is_media].reset_index(drop=True)
print(f"Media dibuang: {is_media.sum()}, sisa {len(df_raw)}")

Media dibuang: 29558, sisa 37333


# 8. Cleaning text

In [13]:
# RT ditulis "RT <teks>" (sering tanpa @user), reply "... [RE username]"
RE_RT_PREFIX = re.compile(r"^\s*RT\b\s*(?:@\w+\s*:\s*)?")
RE_RT = re.compile(r"(?<!\w)RT\s*@(\w+)", re.IGNORECASE)
RE_REPLY = re.compile(r"\[RE\s+@?(\w+)\]", re.IGNORECASE)
RE_ELONGATION = re.compile(r"(.)\1{2,}")
EMOJI_LANG = "id" if "id" in emoji.LANGUAGES else "en"


def strip_noise(text):
    text = RE_REPLY.sub(" ", RE_RT_PREFIX.sub("", text))
    text = emoji.demojize(text, language=EMOJI_LANG).replace("_", " ").replace(":", " ")
    return RE_ELONGATION.sub(r"\1\1", text)


def build_raw_text(headline, mentions):
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if headline.lower() in ("", "nan", mentions.lower()):
        return mentions or headline
    return f"{headline}. {mentions}" if mentions else headline


def dedup_key(texts):
    return texts.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" ")


def match_key(texts, n_words=12):
    # 12 kata pertama: RT kepotong ("...") tetep ketemu aslinya
    return dedup_key(texts).str.split().str[:n_words].str.join(" ")


assert strip_noise("RT @abc: halo [RE def]").strip() == "halo" and strip_noise("RT halo").strip() == "halo"

df_raw["text_raw_combined"] = [build_raw_text(h, m) for h, m in zip(df_raw["Headline"], df_raw["Mentions"])]
df_raw["is_retweet"] = df_raw["text_raw_combined"].map(lambda t: bool(RE_RT_PREFIX.match(t)))
df_raw["text_clean"] = df_raw["text_raw_combined"].map(strip_noise).progress_apply(clean_for_bert)

# copypasta: teks non-RT sama persis dari banyak akun. ditandai, tetep dihitung
key = dedup_key(df_raw["text_clean"])
own = ~df_raw["is_retweet"] & key.str.split().str.len().ge(3)
n_accounts = df_raw[own].groupby(key[own])["author_key"].nunique()
df_raw["copypasta"] = own & key.map(n_accounts).ge(COPYPASTA_MIN_ACCOUNTS)
clusters = n_accounts[n_accounts >= COPYPASTA_MIN_ACCOUNTS].sort_values(ascending=False)
print(f"Copypasta: {len(clusters)} cluster, {df_raw['copypasta'].sum()} baris")
display(clusters.head(10).rename("akun").rename_axis("teks").reset_index())

# dedup cuma buat Gemma
df_clean = (df_raw[df_raw["text_clean"].str.split().str.len() >= 3]
            .assign(dedup_key=lambda d: dedup_key(d["text_clean"]))
            .sort_values("is_retweet", kind="stable")   # post asli didahuluin
            .drop_duplicates(subset="dedup_key").sort_index()
            .drop(columns="dedup_key").reset_index(drop=True))
print(f"Unik: {len(df_clean)} dari {len(df_raw)}")

  0%|          | 0/37333 [00:00<?, ?it/s]

Copypasta: 27 cluster, 136 baris


,teks,akun
0,komitmen pemerintah pemberantasan korupsi komitmen pemerintah pemberantasan korupsi nggak ada kawan ini negara ketegasan prabowo dalam pemberantasan korupsi bukan sekadar retorikasebuah komitmen bahwa kedekatan tidak jadi tameng kalau salah diproses tanpa tebang pilih negara di atas segala kepentingan fokus pada hukum keadilan dan masa depan bangsa,17
1,pemerintah melawan oligarki pemerintah melawan oligarki tekad prabowo subianto tidak terbukti hanya lewat katakata tetapi lewat tindakan nyata dari hinaan hingga difitnah ia tetap berdiri untuk menjaga kedaulatan dan kekayaan nasional di tengah tekanan konsistensi mengerucut menjadi kekuatan melawan oligarki melayani rakyat membangun masa depan yang lebih kuat,11
2,pemerintah melawan oligarki pemerintah melawan oligarki tekad prabowo subianto tetap tegas menghadapi oligarki demi kedaulatan dan kekayaan nasional dari hinaan ke panggung kekuasaan ia ribatkan tekad untuk perubahan nyata,7
3,pemerintah melawan oligarki pemerintah melawan oligarki tekad prabowo tidak tergoyahkan lawan oligarki jaga kedaulatan dan perbaiki nasib bangsa dari dalam kekuasaan ketegasan lahir dari suara rakyat,7
4,pemerintah melawan oligarki pemerintah melawan oligarki tekad prabowo tetap goyah untuk bangsa melawan oligarki menjaga kedaulatan dan memperbaiki negeri dari dalam dari hina ia bangkit sekarang memimpin dengan tetap konsisten ora et labora,7
5,pemerintah melawan oligarki pemerintah melawan oligarki tekad prabowo tetap kukuh menapak lurus melawan oligarki bukan demi ambisi pribadi melainkan kedaulatan bangsa tegas gigih dan tidak pernah goyah di ujung badai,6
6,pegiat media sosial salsa erwina mendesak presiden prabowo subianto untuk kembali melakukan reshuffle kabinet setelah pencopotan menteri keuangan purbaya yudhi sadewa dalam postingan nya salsa erwina mengaku muak dan menuntut presiden prabowo untuk mencopot empat menteri berikut 1 menteri kehutanan raja juli antoni menurut salsa raja juli dinilai gagal menangani kebakaran hutan dan lahan karhutla 2026 yang mengakibatkan 600 ribu hektare lahan terbakar dan 113 ribu warga terkena infeksi saluran pernapasan akut ispa 2 menteri esdm bahlil lahadalia menurut salsa bahlil disorot akibat pemadaman listrik bergilir dan antrean panjang bbm di kalimantan yang terjadi sejak bulan juni 3 menteri komdigi meutya hafid menurut salsa presiden prabowo harus segera mencopot menteri komdigi meutya hafid hal itu karena komdigi memblokir kritik rakyat hingga memblokir media sementara spam judi online justru melonjak hingga 128 persen 4 menteri ham natalius pigai seolah frustrasi dengan pigai salsa mengaku tidak tahu harus membeberkan berbagai kesalahan yang dilakukan menteri ham menurut salsa pigai dianggap telah mengecewakan dan tidak pernah hadir membela rakyat yang tertindas salsa mengaku muak dan meminta presiden prabowo untuk secepatnya mengganti empat menteri tersebut,6
7,pemerintah melawan oligarki pemerintah melawan oligarki tekad prabowo menjaga kedaulatan mengawal kekayaan nasional menghadapi oligarki tanpa surut,6
8,pemerintah melawan oligarki pemerintah melawan oligarki tekad prabowo tidak padam melawan oligarki pengkhianat kini memimpin tegas demi kedaulatan dan kemakmuran nasional dulu dihina difitnah kini bangkit dari rakyat konsistensi di bawah tekanan menegaskan arah menjaga negara memperbaiki keadaan tanpa kompromi,6
9,menunjuk ke arah kanan heboh dpr setuju anggaran bgn 240 triliun tiga hari usai purbaya dicopot dari menkeu komisi ix dpr ri menyetujui pagu alokasi anggaran badan gizi nasional bgn 2027 sebesar rp2402 triliun persetujuan tersebut menjadi salah satu kesimpulan rapat dengar pendapat rdp komisi ix dpr ri bersama kepala bgn dan kepala badan pengawas obat dan makanan bpom kamis 1792026 kesimpulan rdp tersebut dibacakan wakil ketua komisi ix dpr ri yahya zaini pagu anggaran bgn 2027 ditetapkan sebesar rp24020149967800 kata yahya lampu hijau anggaran bgn ini terjadi berselang tiga hari dari pencopotan purbaya yudhi sad

Unik: 21460 dari 37333


# 11. Klasifikasi per cuitan


In [25]:
SENTIMENTS = ("positif", "negatif", "netral", "kontroversi")
STANCES = ("supporter", "moderat", "oposisi", "tidak_jelas")
UNCLEAR = "tidak_jelas"
LABEL_COLS = ["theme_guess", "sentiment", "stance", "theme_confidence"]
THEME_CACHE_PATH = f"{DRIVE_DIR}/theme_cache_{PROJECT}.json"
POST_SCHEMA = {"type": "object",
               "properties": {"primary_theme": {"type": "string", "enum": RELEVANT_THEMES},
                              "sentiment": {"type": "string", "enum": list(SENTIMENTS)},
                              "stance": {"type": "string", "enum": list(STANCES)}},
               "required": ["primary_theme", "sentiment", "stance"]}
THEME_LIST_STR = "\n".join(f"- {t}: {THEME_DESCRIPTIONS[t]}" for t in RELEVANT_THEMES)

# prefix tetap -> llama-cpp nggak baca ulang tiap cuitan
CLASSIFY_PREFIX = f"""Kamu classifier cuitan bahasa Indonesia tentang: {TOPIC}.
Pihak yang dinilai: {STANCE_TARGET}

Daftar tema:
{THEME_LIST_STR}

Untuk cuitan di bawah:
1. membahas_topik: true HANYA kalau isi utamanya tentang peristiwa di topik itu atau dampak langsungnya.
   false kalau membahas isu lain walau menyebut tokoh, negara, acara, atau istilah yang sama.
2. primary_theme: tema paling cocok.
3. sentiment: SIKAP penulis ke topik, BUKAN nada bahasanya.
   positif = mendukung/memuji/membela; negatif = mengkritik/menolak/menyindir;
   netral = hanya informasi; kontroversi = dukungan & kritik sama kuat.
   Membela {STANCE_TARGET} sambil memaki pengkritiknya tetap positif.
4. 4. stance: sikap penulis ke {STANCE_TARGET}. Penentunya siapa yang dibela/diserang, BUKAN nada kalimat.
   supporter = mendukung/memuji/membela, termasuk menyerang pengkritik {STANCE_TARGET};
   oposisi = mengkritik/menolak/menyerang {STANCE_TARGET} atau kebijakannya;
   moderat = menimbang dua sisi, mengkritik kedua kubu, atau kritik tanpa menolak;
   tidak_jelas = hanya informasi/berita/dampak, tanpa memuji atau menyalahkan {STANCE_TARGET}.
   Kalimat kasar/negatif belum tentu oposisi; cek dulu siapa sasarannya.

"""
TRUE_FALSE_IDS = [llm.tokenize(w.encode(), add_bos=False)[0] for w in ("true", "false")]
assert TRUE_FALSE_IDS[0] != TRUE_FALSE_IDS[1]


def p_relevan(content):
    # pancing '{"membahas_topik": ', softmax logit true vs false.
    # pindah ke API LLM? cek dulu logprobs tersedia
    logits = []

    def grab(input_ids, scores):
        logits.append(scores[TRUE_FALSE_IDS].copy())
        return scores

    prompt = f"<start_of_turn>user\n{content}<end_of_turn>\n<start_of_turn>model\n" + '{"membahas_topik": '
    llm.create_completion(prompt, max_tokens=1, temperature=0, logits_processor=llama_cpp.LogitsProcessorList([grab]))
    lt, lf = logits[0]
    return float(1 / (1 + np.exp(lf - lt)))


df_clean[["theme"] + LABEL_COLS] = None

# 1) tanpa keyword -> 'lainnya'. keyword di >20% cuitan diabaikan (terlalu umum)
MAX_KEYWORD_SHARE = 0.2
if RE_RELEVAN:
    found = df_clean["text_clean"].str.lower().map(lambda t: set(RE_RELEVAN.findall(t)))
    counts = pd.Series([k for s in found for k in s], dtype=object).value_counts()
    too_common = set(counts[counts > MAX_KEYWORD_SHARE * len(df_clean)].index)
    if too_common:
        print(f"Keyword diabaikan (terlalu umum): {sorted(too_common)}")
    df_clean.loc[found.map(lambda s: not s - too_common), "theme"] = "lainnya"
review_idx = df_clean.index[df_clean["theme"].isna()].tolist()
if len(review_idx) > 0.5 * len(df_clean):
    raise RuntimeError(f"{len(review_idx)}/{len(df_clean)} lolos prefilter, keyword terlalu umum")

# 2) cache, valid kalau tema, model, & prompt sama
theme_cache = {}
if os.path.exists(THEME_CACHE_PATH):
    with open(THEME_CACHE_PATH, encoding="utf-8") as f:
        cached = json.load(f)
    if (cached.get("themes"), cached.get("model"), cached.get("prompt")) == (THEME_DESCRIPTIONS, LLM_MODEL_FILE, CLASSIFY_PREFIX):
        theme_cache = cached["texts"]
for i in review_idx:
    if hit := theme_cache.get(df_clean.at[i, "text_clean"]):
        df_clean.loc[i, LABEL_COLS] = hit
todo_idx = [i for i in review_idx if pd.isna(df_clean.at[i, "theme_guess"])]
print(f"Prefilter: {len(df_clean) - len(review_idx)} lainnya, {len(review_idx)} ke Gemma ({len(todo_idx)} baru)")

# 3) per cuitan: 1 request (tema+sentimen+stance) + peluang relevan
try:
    for i in tqdm(todo_idx, desc="Gemma"):
        text = df_clean.at[i, "text_clean"]
        content = CLASSIFY_PREFIX + f'Cuitan: "{" ".join(str(text).split()[:120])}"\n\nOutput HANYA JSON.'
        try:
            out = ask_llm(content, POST_SCHEMA, max_tokens=80)
            theme_cache[text] = [out["primary_theme"], out["sentiment"], out["stance"], p_relevan(content)]
        except (ValueError, KeyError) as e:   # jawaban rusak dilewatin
            print(f"Baris {i} skip: {e}")
            continue
        df_clean.loc[i, LABEL_COLS] = theme_cache[text]
finally:
    with open(THEME_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump({"themes": THEME_DESCRIPTIONS, "model": LLM_MODEL_FILE, "prompt": CLASSIFY_PREFIX, "texts": theme_cache},
                  f, ensure_ascii=False)

# ganti threshold di Form -> jalanin ulang cell ini (dari cache)
done = df_clean["theme_guess"].notna()
relevan = df_clean["theme_confidence"].astype(float) >= RANKING_MIN_CONFIDENCE
df_clean.loc[done, "theme"] = df_clean["theme_guess"].where(relevan, "lainnya")[done]
if df_clean["theme"].isna().any():
    print(f"{df_clean['theme'].isna().sum()} baris belum ada tema, jalanin ulang")
print(df_clean["theme"].value_counts(dropna=False).to_string())

Prefilter: 21303 lainnya, 157 ke Gemma (157 baru)


Gemma:   0%|          | 0/157 [00:00<?, ?it/s]

theme
lainnya                                 21341
diplomasi_dan_geopolitik                   51
reaksi_publik_dan_narasi_politik           38
kerja_sama_ekonomi_energi_pertahanan       30


# 13. Edges

In [26]:
HANDLE = re.compile(r"(?<![\w.])@([\w.]*\w)")   # handle IG bertitik ikut


def extract_targets(text, retweet_of=None):
    # 1 edge per akun tujuan, prioritas retweet > reply > mention
    rt = RE_RT.search(text)
    source = canonical(rt.group(1)) if rt else retweet_of
    targets = {source: "retweet"} if source else {}
    for m in RE_REPLY.findall(text):
        targets.setdefault(canonical(m), "reply")
    for m in HANDLE.findall(text):
        targets.setdefault(canonical(m), "mention")
    return list(targets.items())


def undirected(pair_w):
    # A->B + B->A dijumlah
    pair = np.sort(pair_w[["Source", "Target"]].to_numpy(), axis=1)
    und = pair_w.assign(u=pair[:, 0], v=pair[:, 1]).groupby(["u", "v"], as_index=False)["Weight"].sum()
    return nx.from_pandas_edgelist(und, "u", "v", edge_attr="Weight")


def eigenvector_all(U):
    # per klaster, diskala (eigen klaster / eigen terbesar) biar klaster kecil tetep dapet nilai
    parts = []
    for comp in nx.connected_components(U):
        nodes = list(comp)
        A = nx.to_scipy_sparse_array(U, nodelist=nodes, weight="Weight", dtype=float)
        val, vec = eigsh(A, k=1, which="LA", v0=np.ones(len(nodes)))
        parts.append((val[0], pd.Series(np.abs(vec[:, 0]), index=nodes)))
    lam_max = max(lam for lam, _ in parts)
    return pd.concat([vec * lam / lam_max for lam, vec in parts])


def centrality(pair_w):
    G = nx.from_pandas_edgelist(pair_w, "Source", "Target", edge_attr="Weight", create_using=nx.DiGraph)
    out = pd.DataFrame({"pagerank": pd.Series(nx.pagerank(G, weight="Weight"))})
    out["eigenvector"] = eigenvector_all(undirected(pair_w)).reindex(out.index, fill_value=0.0)
    return out.rename_axis("Id").reset_index()


assert extract_targets("RT @Abc: halo @Def @abc. @def, email x@gmail.com") == [("abc", "retweet"), ("def", "mention")]
assert extract_targets("RT halo @def", retweet_of="abc") == [("abc", "retweet"), ("def", "mention")]
assert extract_targets("setuju banget [RE Abc]") == [("abc", "reply")]
_toy = centrality(pd.DataFrame({"Source": [f"r{i}" for i in range(8)] + ["x"],
                                "Target": ["hub"] * 8 + ["y"], "Weight": [1] * 9})).set_index("Id")["eigenvector"]
assert abs(_toy["hub"] - 0.7071) < 1e-3 and abs(_toy["r0"] - 0.25) < 1e-3 and 0 < _toy["x"] < _toy["hub"]

df_topic = df_clean[df_clean["theme"].isin(RELEVANT_THEMES)]
if df_topic.empty:
    raise RuntimeError("Nggak ada cuitan relevan")

# label ke semua baris df_raw: RT ikut label teks yang di-RT
raw_keys = match_key(df_raw["text_clean"])
by_key = df_topic.assign(k=match_key(df_topic["text_clean"])).drop_duplicates("k").set_index("k")
df_raw[["theme", "sentiment", "stance"]] = by_key[["theme", "sentiment", "stance"]].reindex(raw_keys).to_numpy()

# RT tanpa @user: penulis asli = akun pertama yang posting teks itu
is_rt = df_raw["is_retweet"]
orig_author = (df_raw.assign(k=raw_keys)[~is_rt].dropna(subset=["author_key"])
               .drop_duplicates("k").set_index("k")["author_key"])
retweet_of = raw_keys.map(orig_author).where(is_rt)

edge_rows = pd.DataFrame([
    {"row": i, "Source": author, "Target": target, "interaction": kind, "theme": theme}
    for i, author, raw, theme, rt_of in zip(df_raw.index, df_raw["author_key"], df_raw["text_raw_combined"],
                                            df_raw["theme"], retweet_of)
    if pd.notna(theme) and pd.notna(author)
    for target, kind in extract_targets(raw, rt_of if pd.notna(rt_of) else None)
    if target != author
], columns=["row", "Source", "Target", "interaction", "theme"])
df_edges = edge_rows.groupby(["Source", "Target", "interaction", "theme"]).size().rename("Weight").reset_index()

print(f"RT ketemu aslinya: {retweet_of.notna().sum()}/{is_rt.sum()}")
print("Edges:", df_edges.groupby("interaction")["Weight"].sum().to_dict())

RT ketemu aslinya: 4938/13816
Edges: {'mention': 81, 'reply': 9, 'retweet': 11}


# 14. Skor KOL + nodes

In [27]:
EXCLUDE_IDS = TARGET_IDS | {"metaai", "youtube", "instagram", "tiktok", "facebook", "twitter", "threads"}


def is_excluded(key):
    # target, platform, media: tetep node, bukan KOL
    return key in EXCLUDE_IDS or key in MEDIA_IDS or is_media_account(key, extra_accounts=EXTRA_MEDIA_ACCOUNTS)


def top_stance(s):
    # modus stance, 'tidak_jelas' kalau nggak ada sikap sama sekali
    known = s.dropna()[lambda x: x != UNCLEAR]
    return known.mode().iloc[0] if len(known) else UNCLEAR


df_raw["engagement"] = df_raw["Favourited"] + df_raw["Retweeted"]
posted = df_raw[df_raw["theme"].notna() & df_raw["author_key"].notna()]

# volume dari post asli (Retweeted di baris RT punya tweet asli). dari df_raw biar akun copypasta tetep kehitung
agg = (posted[~posted["is_retweet"]].groupby(["theme", "author_key"])
       .agg(jumlah_post=("engagement", "size"), total_engagement=("engagement", "sum"), copypasta=("copypasta", "mean"))
       .reset_index().rename(columns={"author_key": "Id"}))

# bobot per jenis interaksi, log biar 1 akun RT 50x != 50 akun
w_edges = (df_edges.assign(Weight=df_edges["Weight"] * df_edges["interaction"].map(INTERACTION_WEIGHT))
           .groupby(["theme", "Source", "Target"], as_index=False)["Weight"].sum())
w_edges["Weight"] = np.log1p(w_edges["Weight"])
net_parts = [centrality(sub).assign(theme=theme) for theme, sub in w_edges.groupby("theme")]
df_net = pd.concat(net_parts, ignore_index=True) if net_parts else pd.DataFrame(columns=["Id", "pagerank", "eigenvector", "theme"])

display_name = (df_raw.value_counts(["author_key", "Author"]).reset_index()
                .drop_duplicates("author_key").set_index("author_key")["Author"])
followers_by_id = df_raw.groupby("author_key")["Followers"].max()

kol = agg.merge(df_net, on=["theme", "Id"], how="outer")
kol = kol[~kol["Id"].map(is_excluded).astype(bool)].copy()
retweeted = edge_rows[edge_rows["interaction"] == "retweet"]
has_text = set(zip(posted["theme"], posted["author_key"])) | set(zip(retweeted["theme"], retweeted["Target"]))
kol = kol[[p in has_text for p in zip(kol["theme"], kol["Id"])]].copy()
num = ["jumlah_post", "total_engagement", "copypasta", "pagerank", "eigenvector"]
kol[num] = kol[num].fillna(0)
kol["followers"] = kol["Id"].map(followers_by_id).fillna(0)
kol["author_id"] = kol["Id"].map(display_name).fillna(kol["Id"])
kol["score"] = kol.groupby("theme")[SCORE_COLS].rank(pct=True).mean(axis=1)   # percentile: tahan heavy tail

df_top_authors = (kol.sort_values(["theme", "score"], ascending=[True, False])
                  .groupby("theme").head(TOP_N_AUTHORS_PER_THEME).reset_index(drop=True))
df_top_authors["rank"] = df_top_authors.groupby("theme").cumcount() + 1

node_stance = posted.groupby(["theme", "author_key"])["stance"].agg(top_stance)
top_pairs = set(zip(df_top_authors["theme"], df_top_authors["Id"]))
pairs = list(zip(df_net["theme"], df_net["Id"]))
df_nodes = df_net.assign(akun=df_net["Id"].map(display_name).fillna(df_net["Id"]),
                         followers=df_net["Id"].map(followers_by_id),
                         stance=[node_stance.get(p) for p in pairs],
                         is_target=df_net["Id"].isin(TARGET_IDS),
                         is_kol=[p in top_pairs for p in pairs])
print(f"KOL {len(df_top_authors)}, nodes {len(df_nodes)}, edges {len(df_edges)}. Target (di luar ranking):")
display(df_nodes[df_nodes["is_target"]].sort_values("pagerank", ascending=False)[["theme", "akun", "pagerank"]])

KOL 30, nodes 77, edges 101. Target (di luar ranking):


,theme,akun,pagerank
9,diplomasi_dan_geopolitik,prabowo,0.055393
24,reaksi_publik_dan_narasi_politik,prabowo,0.035760
46,reaksi_publik_dan_narasi_politik,@Gerindra,0.015316


# 15. Cek skor & jaringan


In [28]:
# overlap top-N vs skor utama kalau 1 metrik dibuang / jumlah_post ditambah (1 = sama)
pct = kol.groupby("theme")[SCORE_COLS + ["jumlah_post"]].rank(pct=True)


def top_ids(score):
    return (kol.assign(s=score).sort_values("s", ascending=False)
            .groupby("theme").head(TOP_N_AUTHORS_PER_THEME).groupby("theme")["Id"].apply(set))


base = top_ids(kol["score"])
variants = {f"-{c}": [x for x in SCORE_COLS if x != c] for c in SCORE_COLS} | {"+jumlah_post": SCORE_COLS + ["jumlah_post"]}
overlap = {}
for name, cols in variants.items():
    alt = top_ids(pct[cols].mean(axis=1))
    overlap[name] = {t: len(base[t] & alt[t]) / len(base[t]) for t in base.index}
display(pd.DataFrame(overlap).round(2))

# ~1 = pagerank & eigenvector redundan
rho = kol.groupby("theme")[["pagerank", "eigenvector"]].apply(lambda d: d["pagerank"].corr(d["eigenvector"], method="spearman"))
print("Spearman pagerank~eigenvector:", rho.round(2).to_dict())

# assortativity: + = sekubu ngumpul, 0 = campur, - = antar kubu nyambung. louvain: 5 komunitas terbesar x stance
for theme, sub in w_edges.groupby("theme"):
    U = undirected(sub)
    stance = {n: node_stance.get((theme, n)) for n in U}
    nx.set_node_attributes(U, stance, "stance")
    S = U.subgraph([n for n, s in stance.items() if s in STANCES[:3]])
    r = nx.attribute_assortativity_coefficient(S, "stance") if S.number_of_edges() else float("nan")
    comms = sorted(nx.community.louvain_communities(U, weight="Weight", seed=42), key=len, reverse=True)[:5]
    comm = pd.Series({n: f"K{i + 1} ({len(c)})" for i, c in enumerate(comms) for n in c})
    print(f"\n{theme}: assortativity {r:.2f}")
    display(pd.crosstab(comm, comm.index.to_series().map(stance), rownames=["komunitas"], colnames=["stance"]))

,-pagerank,-eigenvector,-total_engagement,-followers,+jumlah_post
diplomasi_dan_geopolitik,0.7,0.7,1.0,0.9,0.6
kerja_sama_ekonomi_energi_pertahanan,1.0,1.0,0.9,0.7,0.9
reaksi_publik_dan_narasi_politik,0.9,0.9,0.9,0.8,1.0


Spearman pagerank~eigenvector: {'diplomasi_dan_geopolitik': 0.99, 'kerja_sama_ekonomi_energi_pertahanan': 1.0, 'reaksi_publik_dan_narasi_politik': 0.88}

diplomasi_dan_geopolitik: assortativity nan


/usr/local/lib/python3.13/dist-packages/networkx/algorithms/assortativity/correlation.py:288: RuntimeWarning: invalid value encountered in scalar divide
  r = (t - s) / (1 - s)


stance,oposisi,supporter,tidak_jelas
komunitas,,,
K1 (7),0,0,7
K2 (3),1,0,0
K3 (3),0,3,0
K4 (2),0,2,0



kerja_sama_ekonomi_energi_pertahanan: assortativity nan


stance,supporter,tidak_jelas
komunitas,,
K1 (2),1,0
K2 (2),0,1



reaksi_publik_dan_narasi_politik: assortativity 0.35


stance,moderat,oposisi,supporter
komunitas,,,
K1 (35),0,2,0
K2 (10),0,3,1
K3 (6),2,0,1
K4 (3),0,1,0
K5 (2),0,2,0


# 16.Stance & ringkasan KOL

In [29]:
SUMMARY_SCHEMA = {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}
KOL_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}

Cuitan & retweet akun "{author}" di tema ini:
{posts}

summary: 2-3 kalimat pandangan/narasi utama akun ini soal tema tersebut, jangan menyalin cuitan.
Bahasa Indonesia. Output HANYA JSON."""

rows_by_kol = posted.groupby(["theme", "author_key"]).groups
rt_rows = edge_rows[edge_rows["interaction"] == "retweet"].groupby(["theme", "Target"])["row"].agg(list).to_dict()

records = []
for r in tqdm(df_top_authors.itertuples(), total=len(df_top_authors), desc="KOL"):
    # cuitan & RT akun ini; kalau kosong, teks dia yang di-RT orang
    rows = df_raw.loc[list(rows_by_kol.get((r.theme, r.Id), [])) or rt_rows.get((r.theme, r.Id), [])]
    stance = top_stance(rows["stance"])
    posts = (rows.sort_values("engagement", ascending=False)["text_clean"]
             .drop_duplicates().head(MAX_POSTS_PER_AUTHOR_SUMMARY).tolist())
    summary = ""
    if posts:
        try:
            summary = ask_llm(KOL_PROMPT.format(topic=TOPIC, theme=r.theme.replace("_", " "), author=r.author_id,
                                                posts=fmt_posts(posts)), SUMMARY_SCHEMA)["summary"].strip()
        except (ValueError, KeyError) as e:
            print(f"{r.author_id} ({r.theme}) gagal: {e}")
    records.append({"theme": r.theme, "rank": r.rank, "author_id": r.author_id, "Id": r.Id, "score": r.score,
                    "jumlah_teks": len(rows),
                    "sentiment": rows["sentiment"].mode().iloc[0] if rows["sentiment"].notna().any() else UNCLEAR,
                    "stance": stance,
                    "stance_share": rows["stance"].eq(stance).mean() if len(rows) else 0.0,   # confidence
                    "copypasta": r.copypasta, "summary": summary})
df_kol = pd.DataFrame(records)

no_text = df_kol[df_kol["jumlah_teks"].eq(0)].groupby("theme").size()
if len(no_text):
    print(f"KOL tanpa teks (cuma di-mention): {no_text.to_dict()}")

KOL:   0%|          | 0/30 [00:00<?, ?it/s]

# 17.Ringkasan per tema

In [30]:
THEME_FIELDS = ["ringkasan", "narasi_supporter", "narasi_moderat", "narasi_oposisi"]
THEME_SCHEMA = {"type": "object", "properties": {f: {"type": "string"} for f in THEME_FIELDS}, "required": THEME_FIELDS}
THEME_PROMPT = """Kamu analis opini publik di media sosial Indonesia.

Topik: {topic}
Tema: {theme}
Pihak yang dinilai: {target}
Komposisi sikap KOL: {composition}

Ringkasan KOL, urut dari paling berpengaruh:
{kols}

Tugas (HANYA dari ringkasan di atas, tanpa fakta luar):
1. ringkasan: 3-4 kalimat isi & arah percakapan di tema ini.
2. narasi_supporter: 1-2 kalimat argumen utama supporter, "-" kalau tidak ada.
3. narasi_moderat: 1-2 kalimat argumen utama moderat, "-" kalau tidak ada.
4. narasi_oposisi: 1-2 kalimat argumen utama oposisi, "-" kalau tidak ada.

Bahasa Indonesia. Output HANYA JSON."""

theme_records = []
for theme, kols in df_kol[df_kol["summary"] != ""].sort_values("rank").groupby("theme"):
    counts = kols["stance"].value_counts()
    content = THEME_PROMPT.format(
        topic=TOPIC, theme=theme.replace("_", " "), target=STANCE_TARGET,
        composition=", ".join(f"{s} {counts.get(s, 0)}" for s in STANCES),
        kols="\n".join(f"- [{k.stance}] {k.author_id}: {k.summary}" for k in kols.itertuples()))
    try:
        res = ask_llm(content, THEME_SCHEMA)
        out = {f: res[f].strip() for f in THEME_FIELDS}
    except (ValueError, KeyError) as e:
        print(f"Tema {theme} gagal: {e}")
        out = dict.fromkeys(THEME_FIELDS, "")
    theme_records.append({"theme": theme, "jumlah_kol": len(kols), **{s: int(counts.get(s, 0)) for s in STANCES}, **out})
df_theme_summary = pd.DataFrame(theme_records)

# Hasil akhir

In [31]:
STANCE_LABEL = {"supporter": "Mendukung", "moderat": "Berimbang", "oposisi": "Menentang", "tidak_jelas": "Tanpa sikap"}
SENT_LABEL = {"positif": "Positif", "negatif": "Negatif", "netral": "Netral", "kontroversi": "Pro & kontra"}
INTERACTION_LABEL = {"retweet": "di-RT", "reply": "dibalas", "mention": "di-mention"}
nice = lambda s: s.str.replace("_", " ").str.capitalize()   # nama tema enak dibaca
name = lambda ids: ids.map(display_name).fillna(ids)

n = df_kol["jumlah_teks"]
view = pd.DataFrame({
    "Tema": nice(df_kol["theme"]),
    "#": df_kol["rank"],
    "Akun": df_kol["author_id"],
    "Skor pengaruh": df_kol["score"].round(2),
    f"Sikap ke {STANCE_TARGET}": df_kol["stance"].map(STANCE_LABEL).where(n > 0, "Nggak ada cuitan"),
    "Dasar sikap": [f"{round(s * k)}/{k} cuitan" if k else "-" for s, k in zip(df_kol["stance_share"], n)],
    "Nada ke topik": df_kol["sentiment"].map(SENT_LABEL).fillna("-"),
    "Ringkasan narasi": df_kol["summary"],
})
if df_kol["copypasta"].gt(0).any():
    view.insert(6, "Copypasta", df_kol["copypasta"].map("{:.0%}".format))
print("Skor pengaruh: rata-rata persentil PageRank, eigenvector, engagement, followers di tema (1 = tertinggi)")
print("Dasar sikap: cuitan yang sikapnya sama / total cuitan yang dibaca")
display(view)

print("\nNada ke topik vs sikap ke target (cuma diagonal = sikap ngikut nada):")
display(pd.crosstab(df_kol["sentiment"].map(SENT_LABEL).fillna("-"), df_kol["stance"].map(STANCE_LABEL),
                    rownames=["nada"], colnames=["sikap"]))

if len(df_theme_summary):
    print("\nRingkasan tema:")
    display(df_theme_summary.assign(theme=nice(df_theme_summary["theme"])).rename(columns={
        "theme": "Tema", "jumlah_kol": "KOL", **STANCE_LABEL, "ringkasan": "Ringkasan",
        "narasi_supporter": "Narasi pendukung", "narasi_moderat": "Narasi berimbang", "narasi_oposisi": "Narasi penentang"}))

print("\nTop 10 akun per tema di jaringan:")
nodes = df_nodes.sort_values(["theme", "pagerank"], ascending=[True, False]).groupby("theme").head(10)
display(pd.DataFrame({"Tema": nice(nodes["theme"]), "Akun": nodes["akun"],
                      "PageRank": nodes["pagerank"].round(3), "Eigenvector": nodes["eigenvector"].round(3),
                      "Followers": nodes["followers"].map(lambda f: "-" if pd.isna(f) else f"{f:,.0f}"),
                      "Sikap": nodes["stance"].map(STANCE_LABEL).fillna("-"),
                      "Peran": np.select([nodes["is_target"], nodes["is_kol"]], ["Target", "KOL"], "")}))

print("\nAkun yang paling banyak diajak interaksi (jumlah akun unik):")
inbound = (edge_rows.assign(jenis=edge_rows["interaction"].map(INTERACTION_LABEL))
           .pivot_table(index=["theme", "Target"], columns="jenis", values="Source", aggfunc="nunique", fill_value=0)
           .assign(total=edge_rows.groupby(["theme", "Target"])["Source"].nunique())
           .rename_axis(columns=None).reset_index()
           .sort_values(["theme", "total"], ascending=[True, False]).groupby("theme").head(10))
display(inbound.assign(theme=nice(inbound["theme"]), Target=name(inbound["Target"]))
        .rename(columns={"theme": "Tema", "Target": "Akun", "total": "Total akun"}))

Skor pengaruh: rata-rata persentil PageRank, eigenvector, engagement, followers di tema (1 = tertinggi)
Dasar sikap: cuitan yang sikapnya sama / total cuitan yang dibaca


,Tema,#,Akun,Skor pengaruh,Sikap ke Presiden Prabowo dan pemerintah,Dasar sikap,Nada ke topik,Ringkasan narasi
0,Diplomasi dan geopolitik,1,@sibambaang,0.98,Tanpa sikap,1/1 cuitan,Netral,"Akun ini menekankan pentingnya fokus pada isu-isu geopolitik global seperti konflik Ukraina, ketegangan di Timur Tengah, dan potensi krisis energi, serta menyoroti peran Presiden Prabowo, Menlu Sugiono, dan Mentri ESDM Bahlil dalam menghadapi tantangan tersebut."
1,Diplomasi dan geopolitik,2,@mkp_english,0.86,Mendukung,1/1 cuitan,Positif,Akun ini menekankan bahwa Indonesia secara terbuka mendukung Rusia dan mengkritik upaya Barat untuk mengisolasi Rusia. Akun ini juga menyoroti hubungan sejarah antara Indonesia dan Rusia serta potensi Indonesia untuk menjadi kekuatan regional yang lebih besar.
2,Diplomasi dan geopolitik,3,@Pororo266,0.85,Mendukung,1/1 cuitan,Positif,"Akun ini menekankan pentingnya konektivitas antara Indonesia dan Rusia dalam berbagai bidang, seperti pelayaran, penerbangan, perdagangan, dan kerja sama ekonomi. Prabowo dianggap sebagai tokoh yang mendorong hubungan bilateral yang lebih erat antara kedua negara."
3,Diplomasi dan geopolitik,4,@KanofskiLoy,0.78,Mendukung,1/1 cuitan,Positif,"Akun ini memuji kunjungan Presiden Prabowo ke Rusia dan pertemuannya dengan Putin, menekankan hubungan persahabatan Indonesia-Rusia dan keberanian Indonesia untuk mendukung Rusia di tengah tekanan Barat."
4,Diplomasi dan geopolitik,5,@EdjaneMaps,0.76,Mendukung,1/1 cuitan,Positif,"Akun ini memuji kunjungan Presiden Prabowo ke Rusia dan pertemuannya dengan Putin, menekankan hubungan persahabatan Indonesia-Rusia dan keberanian Indonesia dalam mendukung Rusia di tengah tekanan Barat. Akun ini juga menyoroti kekuatan Rusia dalam hal wilayah, sumber daya, dan teknologi."
5,Diplomasi dan geopolitik,6,@rafif_af,0.72,Tanpa sikap,1/1 cuitan,Netral,"Akun ini menyoroti situasi geopolitik global yang kompleks dan mendesak, termasuk konflik di Ukraina, Timur Tengah, dan potensi krisis energi, serta menekankan pentingnya peran Presiden Prabowo, Menlu Sugiono, Mentri ESDM Bahlil, dan Wapres Gibran dalam menghadapi tantangan tersebut."
6,Diplomasi dan geopolitik,7,@GlendMarti35542,0.72,Menentang,1/1 cuitan,Negatif,"Akun ini mengkritik narasi yang mengubah pernyataan Presiden Prabowo tentang kemandirian Indonesia menjadi tuduhan kekuatan asing mendikte dan menghisap kekayaan Indonesia. Akun ini menekankan pentingnya kebijakan bebas-aktif yang tidak menuding kekuatan tertentu tanpa data konkret, terutama saat Indonesia bergabung dengan BRICS yang mencakup China dan Rusia."
7,Diplomasi dan geopolitik,8,@adindahan1,0.70,Tanpa sikap,1/1 cuitan,Netral,"Akun ini menekankan pentingnya fokus pada isu-isu geopolitik global seperti konflik Ukraina, ketegangan di Timur Tengah, dan potensi krisis energi, serta menggarisbawahi peran Presiden Prabowo, Menlu Sugiono, dan Mentri ESDM Bahlil dalam menghadapi tantangan tersebut."
8,Diplomasi dan geopolitik,9,@mamacaaa3,0.69,Tanpa sikap,1/1 cuitan,Netral,"Akun ini menyoroti situasi geopolitik global yang memanas, termasuk konflik di Ukraina, Timur Tengah, dan Israel-Palestina, serta potensi krisis energi global. Akun ini juga menyinggung peran Presiden Prabowo, Menlu Sugiono, Mentri ESDM Bahlil, dan Wapres Gibran dalam menghadapi situasi tersebut."
9,Diplomasi dan geopolitik,10,@favcircles,0.69,Tanpa sikap,1/1 cuitan,Netral,"Akun ini menekankan pentingnya fokus pada isu-isu geopolitik global seperti konflik Ukraina, ketegangan di Timur Tengah, dan potensi krisis energi, serta menggarisbawahi peran Presiden Prabowo, Menlu Sugiono, dan Mentri ESDM Bahlil dalam menghadapi tantangan tersebut."



Nada ke topik vs sikap ke target (cuma diagonal = sikap ngikut nada):


sikap,Berimbang,Mendukung,Menentang,Tanpa sikap
nada,,,,
Negatif,0,0,8,0
Netral,2,0,0,8
Positif,0,12,0,0



Ringkasan tema:


,Tema,KOL,Mendukung,Berimbang,Menentang,Tanpa sikap,Ringkasan,Narasi pendukung,Narasi berimbang,Narasi penentang
0,Diplomasi dan geopolitik,10,4,0,1,5,"Pertemuan Presiden Prabowo dengan Putin di Vladivostok menjadi sorotan, dengan KOL menekankan pentingnya diplomasi dan geopolitik global. Beberapa KOL memuji hubungan Indonesia-Rusia dan keberanian Indonesia dalam mendukung Rusia, sementara yang lain menyoroti tantangan geopolitik global seperti konflik Ukraina dan potensi krisis energi. KOL juga menekankan peran Presiden Prabowo dan jajaran kabinet dalam menghadapi tantangan tersebut.","KOL supporter memuji kunjungan Presiden Prabowo ke Rusia dan pertemuannya dengan Putin, menekankan hubungan persahabatan Indonesia-Rusia dan keberanian Indonesia dalam mendukung Rusia di tengah tekanan Barat.",-,"KOL oposisi mengkritik narasi yang mengubah pernyataan Presiden Prabowo tentang kemandirian Indonesia menjadi tuduhan kekuatan asing mendikte dan menghisap kekayaan Indonesia, menekankan pentingnya kebijakan bebas-aktif yang tidak menuding kekuatan tertentu tanpa data konkret."
1,Kerja sama ekonomi energi pertahanan,10,6,0,1,3,"Kunjungan Presiden Prabowo ke Vladivostok dan pertemuannya dengan Vladimir Putin disoroti oleh KOL sebagai peluang kerja sama ekonomi, energi, dan pertahanan antara Indonesia dan Rusia. KOL supporter melihat kunjungan ini sebagai langkah strategis untuk meningkatkan ketahanan ekonomi dan keamanan nasional, sementara KOL oposisi menyoroti potensi dampak negatif terhadap ekonomi Indonesia dan harga minyak bumi.","KOL supporter melihat kunjungan Prabowo ke Rusia sebagai langkah strategis untuk meningkatkan ketahanan ekonomi dan keamanan nasional melalui kerja sama energi, pupuk, dan pertahanan.",-,KOL oposisi menyoroti potensi dampak negatif kunjungan Prabowo ke Rusia terhadap ekonomi Indonesia dan harga minyak bumi.
2,Reaksi publik dan narasi politik,10,2,2,6,0,"Pertemuan Presiden Prabowo dengan Putin di Vladivostok menuai reaksi beragam di media sosial. Oposisi mengkritik kunjungan tersebut sebagai tidak bijaksana dan tidak efektif, sementara pendukung memuji Prabowo sebagai presiden yang berani dan tegas. Moderat menekankan pentingnya fakta dan menghindari penyebaran informasi palsu.","Prabowo dipuji sebagai presiden yang berani dan tegas, dengan langkah-langkahnya seperti menolak IMF dan kerja sama dengan Rusia dianggap sebagai ancaman bagi Amerika Serikat.","Pentingnya fakta dan menghindari penyebaran informasi palsu ditekankan, dengan kritik terhadap kedua kubu yang menyebarkan informasi palsu.","Kunjungan Prabowo ke Vladivostok dan pertemuannya dengan Putin dikritik sebagai tidak bijaksana, tidak efektif, dan bahkan memalukan, dengan sorotan pada ketidakhadiran Prabowo dalam foto pertemuan BRICS dan pidato Prabowo yang dikritik oleh pejabat Rusia."



Top 10 akun per tema di jaringan:


,Tema,Akun,PageRank,Eigenvector,Followers,Sikap,Peran
1,Diplomasi dan geopolitik,@sibambaang,0.260,0.707,"10,857",Tanpa sikap,KOL
5,Diplomasi dan geopolitik,@mkp_english,0.115,0.408,38,Mendukung,KOL
14,Diplomasi dan geopolitik,@Pororo266,0.079,0.289,166,Mendukung,KOL
8,Diplomasi dan geopolitik,@BANGSAygSUJUD,0.066,0.169,"29,848",-,
9,Diplomasi dan geopolitik,prabowo,0.055,0.093,-,-,Target
0,Diplomasi dan geopolitik,@adindahan1,0.043,0.289,144,Tanpa sikap,KOL
2,Diplomasi dan geopolitik,@Bangbatu524917,0.043,0.289,0,Tanpa sikap,
3,Diplomasi dan geopolitik,@cubibubibam,0.043,0.289,12,Tanpa sikap,
4,Diplomasi dan geopolitik,@EdjaneMaps,0.043,0.289,266,Mendukung,KOL
6,Diplomasi dan geopolitik,@favcircles,0.043,0.289,120,Tanpa sikap,KOL



Akun yang paling banyak diajak interaksi (jumlah akun unik):


,Tema,Akun,di-RT,di-mention,dibalas,Total akun
4,Diplomasi dan geopolitik,@sibambaang,6,0,0,6
1,Diplomasi dan geopolitik,@mkp_english,2,0,0,2
0,Diplomasi dan geopolitik,@BANGSAygSUJUD,0,0,1,1
2,Diplomasi dan geopolitik,@Pororo266,1,0,0,1
3,Diplomasi dan geopolitik,prabowo,0,1,0,1
5,Kerja sama ekonomi energi pertahanan,cingkonoha,0,0,1,1
6,Kerja sama ekonomi energi pertahanan,theeconomicinfluenceid,0,1,0,1
41,Reaksi publik dan narasi politik,prabowo,0,5,0,5
9,Reaksi publik dan narasi politik,annascim,0,2,0,2
11,Reaksi publik dan narasi politik,@bangherwin,0,2,0,2


# 18. Label manual

In [32]:
LABEL_PATH = f"{DRIVE_DIR}/labels_{PROJECT}.json"
LABEL_SAMPLE = {"prefilter": 100, "tema": 200, "post": 150}   # tema dibagi 2: tune & test

# isi, jalanin cell ini. label lama tetep kebaca dari Drive. nilai dari teksnya aja, jangan liat hasil Gemma
LABELS_PREFILTER = {   # baris: 1 relevan / 0 bukan
    # 12345: 0,
}
LABELS_TEMA = {        # baris: tema yang bener (boleh "lainnya")
    # 23456: "lainnya",
}
LABELS_POST = {        # baris: ("sentimen", "stance")
    # 34567: ("negatif", "oposisi"),
}

bad = {k: v for k, v in LABELS_TEMA.items() if v not in THEMES}
bad |= {k: v for k, v in LABELS_POST.items() if v[0] not in SENTIMENTS or v[1] not in STANCES}
assert not bad, f"Label nggak valid: {bad}"

saved = {}
if os.path.exists(LABEL_PATH):
    with open(LABEL_PATH, encoding="utf-8") as f:
        saved = json.load(f)
# disimpen per teks, bukan nomor baris
for part, labels in (("prefilter", LABELS_PREFILTER), ("tema", LABELS_TEMA), ("post", LABELS_POST)):
    saved.setdefault(part, {}).update({df_clean.at[i, "text_clean"]: list(v) if isinstance(v, tuple) else v
                                       for i, v in labels.items()})
with open(LABEL_PATH, "w", encoding="utf-8") as f:
    json.dump(saved, f, ensure_ascii=False, indent=2)
print(f"Label: prefilter {len(saved['prefilter'])}, tema {len(saved['tema'])}, post {len(saved['post'])}")


def show_todo(idx, part, title):
    # sampel acak tetap (random_state) yang belum dilabel
    sample = pd.Series(idx).sample(min(LABEL_SAMPLE[part], len(idx)), random_state=42)
    todo = [i for i in sample if df_clean.at[i, "text_clean"] not in saved[part]]
    print(f"\n{title}: {len(todo)} belum")
    for i in todo:
        print(f"{i}: {' '.join(str(df_clean.at[i, 'text_clean']).split()[:60])}")


show_todo(df_clean.index.difference(review_idx), "prefilter", "LABELS_PREFILTER")
show_todo(df_clean.index[df_clean.index.isin(review_idx) & df_clean["theme"].notna()], "tema", f"LABELS_TEMA {THEMES}")
show_todo(df_topic.index, "post", f"LABELS_POST {SENTIMENTS} x {STANCES}")

Label: prefilter 100, tema 100, post 0

LABELS_PREFILTER: 98 belum
20524: ANDA DI PILIH RAKYAT DI BIAYAI RAKYAT ..ANDA INI PRESIDEN CERMINKAN SEBAGAI PRESIDEN
370: bangga indonesiaprabowovira Ividioyoutubeshortsviralfypvidio shorts.
19058: Presiden terbaik wajah tersenyum lebar bermata hati
14120: silampost on TikTok Kapolri Jenderal Polisi Listyo Sigit Prabowo akhirnya menandatangani surat telegram berisi pengisi jabatan wakapolri. Melalui Surat Telegram dengan nomor ST/1764/VII/KEP/2025, orang nomor satu di Korps Bhayangkara itu mengesahkan Komjen Dedi Prasetyo menjadi wakilnya. "Komjen Dedi Prasetyo diangkat dalam jabatan baru sebagai wakapolri," bunyi surat yang ditandatangani hari ini (5/8). Tidak hanya Dedi, Jenderal Sigit merotasi sejumlah perwira
18748: Seneng lihat..nya. jempol ke atas wajah tersenyum lebar bermata hati
11314: tegangan tinggi TEGAS KETUM GCP TUDINGAN TERHADAP DASCO SALAH SASARAN! SIAPA SEBENARNYA YANG HARUS DIPERIKSA? JAKARTA — Suasana politik nasional kembali

# 19. Evaluasi vs label manual

In [33]:
truth = pd.DataFrame([{"key": k, "sentiment_manual": v[0], "stance_manual": v[1]} for k, v in saved["kol"].items()],
                     columns=["key", "sentiment_manual", "stance_manual"])
ev = df_kol.assign(key=df_kol["theme"] + "|" + df_kol["Id"]).merge(truth, on="key")
if ev.empty:
    print("Belum ada label KOL. Isi LABELS_KOL di cell Label manual dulu")
else:
    for kind in ("sentiment", "stance"):
        print(f"\n===== {kind}: {len(ev)} KOL berlabel, kappa {cohen_kappa_score(ev[f'{kind}_manual'], ev[kind]):.2f} =====")
        print(classification_report(ev[f"{kind}_manual"], ev[kind], zero_division=0))
        display(pd.crosstab(ev[f"{kind}_manual"], ev[kind], rownames=["manual"], colnames=["gemma"]))

pre = list(saved["prefilter"].values())
if pre:
    n_pre = len(df_clean) - len(review_idx)
    k, n = sum(pre), len(pre)
    lo, hi = proportion_confint(k, n, method="wilson")   # wilson lebih akurat buat sampel kecil
    missed = k / n * n_pre
    print(f"\nPrefilter: {k}/{n} sampel ternyata relevan ({k / n:.1%}, rentang 95% {lo:.1%}-{hi:.1%})")
    print(f"Perkiraan cuitan relevan yang kebuang: ~{missed:.0f} dari {n_pre:,} (rentang {lo * n_pre:.0f}-{hi * n_pre:.0f})")
    print(f"Perkiraan recall prefilter: {len(df_topic) / (len(df_topic) + missed):.0%}")

# peluang relevan dari Gemma dicek sama label manual
cal = df_clean.loc[review_idx, ["text_clean", "theme_guess", "theme_confidence"]].dropna()
cal = cal.assign(manual=cal["text_clean"].map(saved["tema"]), p=cal["theme_confidence"].astype(float)).dropna()
if cal.empty:
    print("\nBelum ada label tema. Isi LABELS_TEMA di cell Label manual dulu")
else:
    rel_true = cal["manual"].ne("lainnya")
    print(f"\n===== Tema: {len(cal)} cuitan berlabel, {rel_true.sum()} relevan. "
          f"Baseline (tebak 'lainnya' semua): {(~rel_true).mean():.0%} =====")
    # kalau rata_peluang mirip beneran_relevan, peluang dari Gemma bisa dipercaya
    bins = cal.groupby(pd.cut(cal["p"], [0, 0.2, 0.4, 0.6, 0.8, 1.0], include_lowest=True), observed=True)
    display(bins.agg(jumlah=("p", "size"), rata_peluang=("p", "mean"),
                     beneran_relevan=("manual", lambda m: m.ne("lainnya").mean())))

    # threshold naik = yang lolos lebih bersih (precision naik), tapi lebih banyak yang kelewat (recall turun)
    rows = []
    for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
        pred = cal["theme_guess"].where(cal["p"] >= t, "lainnya")
        rel_pred = pred.ne("lainnya")
        rows.append({"threshold": t, "lolos": int(rel_pred.sum()),
                     "precision": precision_score(rel_true, rel_pred, zero_division=0),
                     "recall": recall_score(rel_true, rel_pred, zero_division=0),
                     "f1": f1_score(rel_true, rel_pred, zero_division=0),
                     "akurasi_tema": (pred == cal["manual"]).mean()})
    thr = pd.DataFrame(rows)
    display(thr)
    best = thr.loc[thr["f1"].idxmax()]
    print(f"F1 tertinggi di threshold {best['threshold']} (precision {best['precision']:.0%}, recall {best['recall']:.0%}). "
          f"Sekarang {RANKING_MIN_CONFIDENCE}. Mau ganti? Ubah RANKING_MIN_CONFIDENCE di Form, lalu jalanin ulang dari Klasifikasi tema")
    # salahnya ke mana di threshold sekarang (baris = label manual, kolom = tema akhir)
    display(pd.crosstab(cal["manual"], df_clean.loc[cal.index, "theme"], rownames=["manual"], colnames=["gemma"]))

Belum ada label KOL. Isi LABELS_KOL di cell Label manual dulu

Prefilter: 0/100 sampel ternyata relevan (0.0%, rentang 95% 0.0%-3.7%)
Perkiraan cuitan relevan yang kebuang: ~0 dari 21,303 (rentang 0-788)
Perkiraan recall prefilter: 100%

===== Tema: 21 cuitan berlabel, 5 relevan. Baseline (tebak 'lainnya' semua): 76% =====


,jumlah,rata_peluang,beneran_relevan
p,,,
"(-0.001, 0.2]",3,0.056763,0.000000
"(0.2, 0.4]",2,0.285007,0.000000
"(0.4, 0.6]",1,0.596294,1.000000
"(0.6, 0.8]",3,0.741068,0.000000
"(0.8, 1.0]",12,0.931208,0.333333


,threshold,lolos,precision,recall,f1,akurasi_tema
0,0.2,18,0.277778,1.0,0.434783,0.142857
1,0.3,17,0.294118,1.0,0.454545,0.190476
2,0.4,16,0.312500,1.0,0.476190,0.238095
3,0.5,16,0.312500,1.0,0.476190,0.238095
4,0.6,15,0.266667,0.8,0.400000,0.238095
5,0.7,15,0.266667,0.8,0.400000,0.238095
6,0.8,12,0.333333,0.8,0.470588,0.380952
7,0.9,10,0.300000,0.6,0.400000,0.428571


F1 tertinggi di threshold 0.4 (precision 31%, recall 100%). Sekarang 0.5. Mau ganti? Ubah RANKING_MIN_CONFIDENCE di Form, lalu jalanin ulang dari Klasifikasi tema


gemma,diplomasi_dan_geopolitik,kerja_sama_ekonomi_energi_pertahanan,lainnya,reaksi_publik_dan_narasi_politik
manual,,,,
agenda_pertemuan,2,0,0,0
kerja_sama_bilateral,1,1,0,0
lainnya,9,1,5,1
reaksi_geopolitik,1,0,0,0


In [34]:
# sentimen & stance per cuitan
post = df_clean.assign(manual=df_clean["text_clean"].map(saved["post"])).dropna(subset=["manual", "stance"])
if post.empty:
    print("Belum ada LABELS_POST")
for j, kind in enumerate(("sentiment", "stance") if len(post) else ()):
    truth = post["manual"].str[j]
    print(f"\n{kind}: {len(post)} cuitan, kappa {cohen_kappa_score(truth, post[kind]):.2f}")
    print(classification_report(truth, post[kind], zero_division=0))
    display(pd.crosstab(truth, post[kind], rownames=["manual"], colnames=["gemma"]))

# relevan & tema: threshold dipilih di tune, angka dilaporkan di test
cal = df_clean.loc[review_idx, ["text_clean", "theme_guess", "theme_confidence"]].dropna()
cal = cal.assign(manual=cal["text_clean"].map(saved["tema"]), p=cal["theme_confidence"].astype(float)).dropna()
predict = lambda d, t: d["theme_guess"].where(d["p"] >= t, "lainnya")
prec = rec = None
if len(cal) < 20:
    print(f"\nLABELS_TEMA baru {len(cal)}, minimal 20")
else:
    tune = cal.sample(frac=0.5, random_state=42)
    test = cal.drop(tune.index)
    rows = []
    for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
        y, yp = tune["manual"].ne("lainnya"), predict(tune, t).ne("lainnya")
        rows.append({"threshold": t, "precision": precision_score(y, yp, zero_division=0),
                     "recall": recall_score(y, yp, zero_division=0), "f1": f1_score(y, yp, zero_division=0)})
    thr = pd.DataFrame(rows)
    print(f"\nTune ({len(tune)}):")
    display(thr.round(2))
    print(f"F1 terbaik {thr.loc[thr['f1'].idxmax(), 'threshold']} (sekarang {RANKING_MIN_CONFIDENCE})")

    pred = predict(test, RANKING_MIN_CONFIDENCE)
    y, yp = test["manual"].ne("lainnya"), pred.ne("lainnya")
    prec, rec = precision_score(y, yp, zero_division=0), recall_score(y, yp, zero_division=0)
    print(f"Test ({len(test)}): precision {prec:.0%}, recall {rec:.0%}, akurasi tema {(pred[y & yp] == test['manual'][y & yp]).mean():.0%}")
    display(pd.crosstab(test["manual"], pred, rownames=["manual"], colnames=["gemma"]))

# prefilter & recall end-to-end
if pre := list(saved["prefilter"].values()):
    n_pre = len(df_clean) - len(review_idx)
    k, n = sum(pre), len(pre)
    lo, hi = proportion_confint(k, n, method="wilson")
    missed = k / n * n_pre
    print(f"\nPrefilter: {k}/{n} relevan kebuang, ~{missed:.0f}/{n_pre:,} (95%: {lo * n_pre:.0f}-{hi * n_pre:.0f})")
    if rec:
        kept_true = len(df_topic) * prec   # lolos & beneran relevan
        print(f"Recall end-to-end: ~{kept_true / (kept_true / rec + missed):.0%}")

Belum ada LABELS_POST

Tune (10):


,threshold,precision,recall,f1
0,0.2,0.33,1.00,0.50
1,0.3,0.33,1.00,0.50
2,0.4,0.33,1.00,0.50
3,0.5,0.33,1.00,0.50
4,0.6,0.25,0.67,0.36
5,0.7,0.25,0.67,0.36
6,0.8,0.40,0.67,0.50
7,0.9,0.50,0.67,0.57


F1 terbaik 0.9 (sekarang 0.5)
Test (11): precision 29%, recall 100%, akurasi tema 0%


gemma,diplomasi_dan_geopolitik,kerja_sama_ekonomi_energi_pertahanan,lainnya
manual,,,
kerja_sama_bilateral,0,1,0
lainnya,4,1,4
reaksi_geopolitik,1,0,0



Prefilter: 0/100 relevan kebuang, ~0/21,303 (95%: 0-788)
Recall end-to-end: ~100%
